# 02. Tiền Xử Lý Hình Ảnh Nâng Cao & Anti-Data-Leakage Pipeline

Notebook này thử nghiệm và minh họa toàn bộ pipeline tiền xử lý ảnh biển báo giao thông:
1. **Resize ảnh** về kích thước chuẩn (32x32 pixels).
2. **Cân bằng tương phản thích ứng CLAHE** (Contrast Limited Adaptive Histogram Equalization) trên kênh độ sáng $L$ của không gian màu LAB để tối ưu độ chi tiết khi biển báo bị thiếu sáng, chói nắng hoặc bị bóng râm che khuất.
3. **Chuẩn hóa giá trị pixel** về dải $[0.0, 1.0]$.
4. **Phân chia tập dữ liệu chống rò rỉ (Anti Data Leakage)**: Train (70%) / Validation (15%) / Test (15%) được thực hiện *trước* khi tăng cường dữ liệu.
5. **Tăng cường dữ liệu (Data Augmentation)**: Áp dụng xoay nhẹ, dịch chuyển, thay đổi độ sáng/tương phản và giả lập điều kiện thời tiết (mờ/nhiễu) *duy nhất* trên tập Train (Tuyệt đối không lật ngang ảnh để giữ nguyên ý nghĩa biển báo chỉ hướng).

In [ ]:
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
from src.data_loader import load_raw_dataset, find_dataset_dir
from src.preprocessing import (
    split_data, resize_image, apply_clahe, normalize_pixels, augment_single_image,
    save_preprocessing_visualization
)

In [ ]:
# 1. Nạp dữ liệu gốc
dataset_dir, is_sample = find_dataset_dir()
images_raw, labels, stats = load_raw_dataset(dataset_dir=dataset_dir)
print(f"Tổng số ảnh nạp vào: {len(images_raw)} ảnh")

In [ ]:
# 2. Thử nghiệm Cân bằng Tương phản Thích ứng CLAHE (Contrast Limited Adaptive Histogram Equalization)
sample_img = images_raw[0]
resized_img = resize_image(sample_img, (32, 32))
clahe_img = apply_clahe(resized_img, clip_limit=2.0, tile_grid_size=(8, 8))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(resized_img)
ax1.set_title("Ảnh Resize Gốc (32x32)", fontsize=11, fontweight='bold')
ax1.axis('off')

ax2.imshow(clahe_img)
ax2.set_title("Ảnh Sau Khi Áp Dụng CLAHE (Cải Thiện Tương Phản)", fontsize=11, fontweight='bold')
ax2.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Phân chia tập dữ liệu chuẩn 70% Train / 15% Validation / 15% Test (Chống Data Leakage)
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = split_data(
    images_raw, labels, test_ratio=0.15, val_ratio=0.15, random_state=42
)

print(f"Tập Train: {len(X_train_raw)} mẫu ({len(X_train_raw)/len(images_raw)*100:.1f}%)")
print(f"Tập Validation: {len(X_val_raw)} mẫu ({len(X_val_raw)/len(images_raw)*100:.1f}%)")
print(f"Tập Test: {len(X_test_raw)} mẫu ({len(X_test_raw)/len(images_raw)*100:.1f}%)")

In [ ]:
# 4. Minh họa quy trình Tiền Xử Lý 5 bước toàn diện
save_path = save_preprocessing_visualization(images_raw[0])
print(f"Đã xuất hình ảnh trực quan hóa tiền xử lý ra: {save_path}")

## Nguyên Tắc Chống Rò Rỉ Dữ Liệu (Data Leakage Prevention)

1. **Phân chia trước Augmentation:** Việc chia tập Train, Validation và Test được thực hiện **trước** khi áp dụng bất kỳ kỹ thuật Data Augmentation nào. Điều này đảm bảo dữ liệu ảnh biến thể tuyệt đối không bị lọt sang tập Test.
2. **Augmentation chỉ áp dụng cho Tập Train:** Chỉ áp dụng xoay nhẹ, dịch chuyển, thay đổi độ sáng và giả lập nhiễu cho tập Train. Tập Validation và Test được giữ nguyên hình ảnh chuẩn để đánh giá khách quan.
3. **Không lật ngang (No Horizontal Flip):** Biển báo giao thông chứa thông tin chỉ hướng (Rẽ trái / Rẽ phải, Đi bên trái / Đi bên phía phải). Lật ngang ảnh sẽ làm thay đổi bản chất nhãn của biển báo (Ví dụ: biển P.102 Cấm đi ngược chiều hoặc biển chỉ hướng Rẽ trái biến thành Rẽ phải).